# Lausanne Postal simulation demo

This compact tutorial uses the complete prepared Lausanne environment and the production simulation/exposure pipeline. It runs one synthetic full-day replication with one-hour reporting bins, selects ten physical vehicles with a fixed random seed, compares their temporal and spatial sensing duration, and animates one vehicle's trajectory.

Supply uses four Auto service areas. Expected population-weighted origin demand is partitioned before the replication; the 20-vehicle catalog is then frozen across the areas by largest-remainder demand allocation. The depot does not determine membership.

Run every cell in order with Python 3.12 and the repository's optimization/notebook dependencies. No dataset, image, animation or project is saved; the final cell removes the temporary workspace. The display sample does not change operational supply.

In [ ]:
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        "This tutorial requires Python 3.12. Select the "
        "'Mobile Sensing (Python 3.12)' kernel and restart the notebook."
    )

# Prefer this checkout when the notebook is opened from the repository. An
# installed wheel remains valid when no checkout is present.
repository = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "src" / "mobile_sensing").is_dir()
    ),
    None,
)
if repository is not None:
    source_root = str(repository / "src")
    if source_root not in sys.path:
        sys.path.insert(0, source_root)

from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
from IPython.display import HTML, display

from mobile_sensing.application.project_models import ProjectConfig
from mobile_sensing.application.run_models import RunOptions
from mobile_sensing.application.tutorial_workflow import (
    open_tutorial, simulation_result, sampled_vehicle_sensing, vehicle_trajectory,
)
from mobile_sensing.application.tutorial_plots import (
    animate_vehicle_trajectory, display_figure,
    plot_sampled_vehicle_space, plot_sampled_vehicle_time,
)

plt.rcParams.update({'font.size': 10, 'figure.dpi': 110})
temporary = TemporaryDirectory(prefix='mobile-sensing-fleet-tutorial-')
workspace = Path(temporary.name)
bundle, source_run, _ = open_tutorial(workspace)
options = RunOptions(workers=1, memory_limit_bytes=8 * 1024**3, job_timeout_s=14400)


## 1. Configure one full-day replication

Simulation reporting remains **60 minutes**: it controls exposure time bins and the temporal chart. This notebook does not run Portfolio analysis; the main Lausanne tutorial separately sets `utility_temporal_resolution_minutes=1440`, so utility is evaluated once per full day.

In [ ]:
fleet = next(item for item in source_run.config.fleets if item.fleet_id == 'postal')
demand = fleet.demand.model_copy(update={"task_volume": 400.0})
fleet = fleet.model_copy(update={'demand': demand})
simulation = source_run.config.simulation.model_copy(update={
    'replications': 1, 'start_time': '00:00', 'end_time': '24:00',
    'temporal_resolution_minutes': 60.0, 'seed': 20260114,
})
configuration = source_run.config.model_copy(update={
    'schema_version': '3.4', 'fleets': (fleet,), 'simulation': simulation,
})
configuration = ProjectConfig.model_validate_json(configuration.model_dump_json())
display({
    'fleet': fleet.name,
    'tasks': demand.task_volume,
    'physical vehicles': fleet.supply.fleet_size,
    'dispatch': fleet.dispatch.mode,
    'service areas': fleet.supply.service_area_mode,
    'auto area count': fleet.supply.auto_service_area_count,
    'reporting minutes': simulation.temporal_resolution_minutes,
})


## 2. Run operations and exposure

The call resolves the frozen physical catalog and any service areas, executes the common event kernel, then allocates sparse physical-vehicle × grid × hour exposure. A one-replication result is suitable for a transparent demo; it is not an estimate of operational variability.

In [ ]:
run = simulation_result(workspace, configuration, None, options=options)
vehicle_ids, spatial, temporal, environment = sampled_vehicle_sensing(
    workspace, run, 'postal', count=10, seed=20260116,
)
display({'run_id': run.run_id, 'realized tasks': run.task_counts['postal'][0],
         'display vehicles': vehicle_ids})


## 3. Ten sampled vehicles: temporal sensing

Each line is one physical vehicle. Values are sensing minutes in the one-hour reporting bin; zero remains a meaningful inactive or uncovered interval.

In [ ]:
figure = plot_sampled_vehicle_time(
    temporal, title='Lausanne Postal simulation demo' + ' · ten sampled vehicles',
)
display_figure(figure)
plt.close(figure)


## 4. Ten sampled vehicles: full-day spatial sensing

Every panel sums that vehicle's 24 hourly exposure matrices. All ten panels share one square-root color scale, retain zero cells and use the same Lausanne boundary.

In [ ]:
figure = plot_sampled_vehicle_space(
    spatial, environment, title='Lausanne Postal simulation demo' + ' · full-day sensing',
)
display_figure(figure)
plt.close(figure)


## 5. One vehicle's daily trajectory

The animation advances in 15-minute frames. The blue line is cumulative routed movement and the orange marker is the current moving position. Stationary service, waiting and idling can contribute operating-duration sensing but do not create movement geometry.

In [ ]:
for animated_vehicle in vehicle_ids:
    trajectory, trajectory_environment = vehicle_trajectory(
        workspace, run, 'postal', animated_vehicle,
    )
    if not trajectory.empty:
        break
animation = animate_vehicle_trajectory(
    trajectory, trajectory_environment, vehicle_id=animated_vehicle, frame_minutes=15,
)
display(HTML(animation.to_jshtml()))
plt.close(animation._fig)


## 6. Finish

Delete the temporary backend. Restart from the setup cell to run the tutorial again.

In [ ]:
temporary.cleanup()
